In [2]:
from datetime import datetime,date
from typing import Union
Datelike =Union[datetime,date,str]

def _to_date(d:DateLike) ->date:
    if isinstance(d,date) and not isinstance(d,datetime):
        return d
    if isinstance(d,datetime):
        return d.date()
    return datetime.strptime(d,"%Y-%m-%d").date()

def price_model(
        injectiondate:List[tuple[DateLike,float]],
        withdrawldate:List[tuple[DateLike,float]],
        marketprice:Dict[DateLike,float],
        max_storage:float,
        injection_rate:float,
        withdrawal_rate:float,
        Storage_costperday:float=0.0,
        injection_costPermmbtu:float=0.0,
        withdrawal_costPermmbtu:float=0.0) -> float:

        events=[]
        for d,vol in injectiondate:
             events.append((_to_date(d),vol,'injection'))
        for d,vol in withdrawldate:
             events.append((_to_date(d),vol,'withdrawal'))
        events.sort(key=lambda x:(x[0],0 if x[2]=='injection' else 1))
        prices={_to_date(k):float(v) for k,v in marketprice.items()}
        inventory=0.0
        value=0.0
        last_date=None
        for td,type,vol in events:
            if last_date is not None:
                days=(td-last_date).days
                if days<0:
                    raise ValueError("Events are not in order")
                value-=Storage_costperday*days
            if td not in prices:
                raise KeyError("Market price is not given for the date")
            price=prices[td]
            if type=='injection':
                if vol>injection_rate:
                    raise ValueError("Injection exceeds max volume")
                if inventory+vol>max_storage:
                    raise ValueError('Storage is FULL NOW')
                value-=vol*price
                valur-=vol *injection_costPermmbtu
                invenntory+=vol
            else:
                if vol>withdrawal_rate:
                    raise ValueError("Withdrawl exceeds max withdrawl rate")
                if vol>inventory:
                    raise ValueError("Not enough volume available")
                value+=vol*price
                value-=vol*withdrawal_costPermmbtu
                invrntory-=vol
            last_date=td
        if abs(inventory)>0:
            raise ValueError("Inventory is not empty at the end")
        return value           
            
                  



In [4]:
injections = [("2026-01-01", 1_000_000)]
withdrawals = [("2026-05-01", 1_000_000)]
prices = {
    "2026-01-01": 2.0,
    "2026-05-01": 3.0
}

contract_value = price_model(
    injectiondate=injections,
    withdrawldate=withdrawals,
    marketprice=prices,
    max_storage=2_000_000,
    injection_rate=1_000_000,
    withdrawal_rate=1_000_000,
    Storage_costperday=100_000 / 30,
    injection_costPermmbtu=5000 / 1_000_000,
    withdrawal_costPermmbtu=5000 / 1_000_000
)

print(contract_value)

TypeError: '>' not supported between instances of 'str' and 'int'